# Phase 4b-alt: Difference-of-Means Steering Vectors (no training)

**Pivot** from paper's gradient-based training (requires backward through 25 GDN/attention layers — doesn't fit 96GB VRAM) to **difference-of-means** approximation.

**For each cluster c**:
1. Take top-200 sentences activating cluster c (from Phase 4a)
2. For each, reconstruct `prompt+prefix+target`, forward through Qwen3.5-35B-A3B-**Base**, capture L15 activations at target-sentence token positions, mean-pool
3. Global reference: 500 random sentences across clusters, same procedure
4. `v_c = mean(cluster_c) - mean(global)` → steering direction

**Trade-off vs paper**: no gradient-trained vectors. Expected performance: between random-vector (paper ~77.2%) and full-trained (84.4%). Pipeline validation proof-of-concept.

**Budget**: ~30-45 min, 0 backward passes (forward only → fits 96GB easily).

## Cell 1 — Install + setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', 'causal-conv1d', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/phase4b_alt'); OUT.mkdir(exist_ok=True)
HF_PHASE2 = 'caiovicentino1/qwen35-a3b-sae-phase2'
MODEL_ID = 'Qwen/Qwen3.5-35B-A3B-Base'
STEERING_LAYER = 15
N_LATENTS = 15
PER_CLUSTER = 200  # top-200 per cluster
N_RANDOM = 500     # global random baseline
print('setup done')

## Cell 2 — Load Qwen3.5-35B-A3B-Base (no grad tracking)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download, HfApi

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

## Cell 3 — Load steering data + install L15 capture hook

In [ ]:
path = snapshot_download(HF_PHASE2, repo_type='model', cache_dir='/content/cache')
with open(Path(path) / f'steering_data_n{N_LATENTS}.json') as f:
    steering_data = json.load(f)
with open(Path(path) / f'cluster_labels_n{N_LATENTS}.json') as f:
    cluster_labels = json.load(f)

# Hook: capture L15 residual output at all token positions
captured_l15 = {'value': None}
def capture_hook(module, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    captured_l15['value'] = h.detach().clone()  # [B, T, d_model]

h_capture = get_layer_module(model, STEERING_LAYER).register_forward_hook(capture_hook)
D = 2048
print(f'✅ L{STEERING_LAYER} capture hook installed')

## Cell 4 — Extract target-sentence L15 activations per cluster

In [ ]:
MAX_SEQ_LEN = 2048

def get_target_activation(entry):
    """Forward 'prompt + prefix + target' through base. Return mean L15 activation over target tokens."""
    prompt = entry['prompt']
    prefix = entry.get('prefix', '')
    target = entry['target']
    if prefix:
        ctx = f"{prompt} {prefix}"
    else:
        ctx = prompt
    full = f"{ctx} {target}"
    ctx_ids = tok(ctx, add_special_tokens=True, return_tensors='pt').input_ids[0]
    full_ids = tok(full, add_special_tokens=True, return_tensors='pt').input_ids[0]
    if full_ids.shape[0] > MAX_SEQ_LEN: return None
    if full_ids.shape[0] <= ctx_ids.shape[0]: return None
    # Token range of target: [ctx_len, full_len)
    target_start = ctx_ids.shape[0]
    input_ids = full_ids.unsqueeze(0).cuda()
    with torch.no_grad():
        _ = model(input_ids=input_ids, attention_mask=torch.ones_like(input_ids))
    l15 = captured_l15['value'][0]  # [T, d]
    target_l15 = l15[target_start:, :]
    if target_l15.shape[0] == 0: return None
    return target_l15.mean(dim=0).float().cpu().numpy()

# Per-cluster extraction
cluster_activations = {}  # cid -> [N_sentences, d]
t0 = time.time()
for cid in range(N_LATENTS):
    label = cluster_labels[str(cid)]['title']
    entries = steering_data[str(cid)][:PER_CLUSTER]
    print(f'\n=== Cluster {cid}: {label} ({len(entries)} entries) ===')
    acts = []
    for e in tqdm(entries, desc=f'c{cid}', leave=False):
        try:
            a = get_target_activation(e)
            if a is not None: acts.append(a)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); continue
        except Exception as e:
            continue
    cluster_activations[cid] = np.stack(acts) if acts else np.zeros((0, D))
    print(f'  collected {len(acts)} activations')

print(f'\n✅ all clusters in {(time.time()-t0)/60:.1f} min')

# Global random baseline
print('\n=== Global random baseline ===')
all_entries = []
for cid_str, entries in steering_data.items():
    all_entries.extend(entries)
random.seed(42); random.shuffle(all_entries)
random_acts = []
for e in tqdm(all_entries[:N_RANDOM], desc='random'):
    try:
        a = get_target_activation(e)
        if a is not None: random_acts.append(a)
    except: torch.cuda.empty_cache(); continue
global_mean = np.stack(random_acts).mean(axis=0) if random_acts else np.zeros(D)
print(f'✅ global mean from {len(random_acts)} random samples')

## Cell 5 — Compute difference-of-means steering vectors

In [ ]:
steering_vectors = {}
print('Cluster steering vectors (v_c = mean_cluster - mean_global):\n')
print(f'{"cid":>3} | {"label":35s} | {"norm":>6} | {"n_samples":>9}')
for cid in range(N_LATENTS):
    acts = cluster_activations[cid]
    if acts.shape[0] == 0: 
        steering_vectors[cid] = None
        print(f'{cid:>3} | (empty)')
        continue
    cluster_mean = acts.mean(axis=0)
    v_c = cluster_mean - global_mean
    steering_vectors[cid] = v_c
    label = cluster_labels[str(cid)]['title'][:33]
    print(f'{cid:>3} | {label:35s} | {np.linalg.norm(v_c):>6.3f} | {acts.shape[0]:>9}')

# Bias = global mean itself (add bias means shift all activations toward mean thinking distribution)
steering_vectors['bias'] = global_mean
print(f'\nBias (global mean): norm {np.linalg.norm(global_mean):.3f}')

# Save
for k, v in steering_vectors.items():
    if v is None: continue
    fn = OUT / f'v_c{k}.npz'
    np.savez(fn, vector=v)
print(f'\n✅ saved {len([v for v in steering_vectors.values() if v is not None])} vectors')

## Cell 6 — Upload to HF

In [ ]:
api = HfApi()
uploaded = 0
for k in list(steering_vectors.keys()):
    if steering_vectors[k] is None: continue
    path = OUT / f'v_c{k}.npz'
    if path.exists():
        try:
            api.upload_file(path_or_fileobj=str(path),
                            path_in_repo=f'steering_vectors_diffmeans/v_c{k}.npz',
                            repo_id=HF_PHASE2, repo_type='model',
                            commit_message=f'Phase 4b-alt: difference-of-means steering vector {k}')
            uploaded += 1
        except Exception as e:
            print(f'upload {k}: {e}')

# Summary
summary = {}
for k, v in steering_vectors.items():
    if v is None: continue
    summary[str(k)] = {
        'label': cluster_labels.get(str(k), {}).get('title', 'bias') if str(k) != 'bias' else 'global_mean_bias',
        'norm': float(np.linalg.norm(v)),
        'method': 'difference-of-means',
        'n_samples_cluster': int(cluster_activations.get(k, np.zeros((0,))).shape[0]) if k != 'bias' else int(len(random_acts)),
    }
with open(OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
api.upload_file(path_or_fileobj=str(OUT / 'summary.json'),
                path_in_repo='steering_vectors_diffmeans/summary.json',
                repo_id=HF_PHASE2, repo_type='model')
print(f'\n✅ {uploaded} vectors uploaded to https://huggingface.co/{HF_PHASE2}/tree/main/steering_vectors_diffmeans')